In [1]:
import torch
from torch import Tensor
from typing import List, Tuple

In [2]:
class MModule(torch.nn.Module):
  def __init__(self):
      super(MModule, self).__init__()
      self.freqs_cis = self.precompute_freqs_cis(1024, 512)

  def precompute_freqs_cis(self,
  seq_len: int, n_elem: int, base: int = 10000
    ) -> Tensor:
        freqs = 1.0 / (base ** (torch.arange(0, n_elem, 2)[: (n_elem // 2)].float() / n_elem))
        t = torch.arange(seq_len, device=freqs.device)
        freqs = torch.outer(t, freqs)
        freqs_cis = torch.polar(torch.ones_like(freqs), freqs)
        cache = torch.stack([freqs_cis.real, freqs_cis.imag], dim=-1)
        return cache.to(dtype=torch.bfloat16)
  
  def apply_rotary_emb(self, x: Tensor) -> Tensor:
    xshaped = x.float().reshape(*x.shape[:-1], -1, 2)
    freqs_cis = self.freqs_cis.view(1, xshaped.size(1), 1, xshaped.size(3), 2)
    x_out2 = torch.stack(
        [
            xshaped[..., 0] * freqs_cis[..., 0] - xshaped[..., 1] * freqs_cis[..., 1],
            xshaped[..., 1] * freqs_cis[..., 0] + xshaped[..., 0] * freqs_cis[..., 1],
        ],
        -1,
    )

    x_out2 = x_out2.flatten(3)
    return x_out2.type_as(x)

  def forward(self, input: Tensor) -> Tensor:
    return self.apply_rotary_emb(input)


In [3]:
class MoldModuole(torch.nn.Module):
  def __init__(self):
    super(MoldModuole, self).__init__()
    self.freqs_cis = self.precompute_freqs_cis_ted(1024, 512)

  def precompute_freqs_cis_ted(self,
    seq_len: int, n_elem: int, base: int = 10000, device = None
    ) -> torch.Tensor:
    freqs = 1.0 / (base ** (torch.arange(0, n_elem, 2, device=device)[: (n_elem // 2)].float() / n_elem))
    t = torch.arange(seq_len, device=freqs.device)
    freqs = torch.outer(t, freqs)
    freqs_cis = torch.polar(torch.ones_like(freqs), freqs)
    cache = torch.stack([freqs_cis.real, freqs_cis.imag], dim=-1)
    return cache.to(dtype=torch.bfloat16)
  
  def reshape_for_broadcast_new(self, freqs_cis: torch.Tensor, x: torch.Tensor):
    ndim = x.ndim
    # shape = [d if i == 1 or i == ndim - 1 else 1 for i, d in enumerate(x.shape)]
    return freqs_cis.view(1, x.shape[1], 1, x.shape[-1])

  def _rotate(self, x: torch.Tensor, sin: torch.Tensor, cos: torch.Tensor) -> torch.Tensor:
    x_real = x[..., 0::2]
    x_imag = x[..., 1::2]

    freq_cos = self.reshape_for_broadcast_new(cos, x_real).float()
    freq_sin = self.reshape_for_broadcast_new(sin, x_imag).float()

    x_out_real = x_real * freq_cos - x_imag * freq_sin
    x_out_imag = x_real * freq_sin + x_imag * freq_cos
    x_out = torch.stack([x_out_real, x_out_imag], dim=-1).flatten(-2)
    return x_out.type_as(x)


  def cos_sin_apply_rotary_emb_new(
    self,
    xq: torch.Tensor,
    xk: torch.Tensor,
    freqs_cis: torch.Tensor,
  ) -> Tuple[torch.Tensor, torch.Tensor]:
    sin, cos = freqs_cis[..., 0], freqs_cis[..., 1]
    return self._rotate(xq, sin, cos), self._rotate(xk, sin, cos)
  
  def forward(self, input: Tensor) -> Tensor:
    return self.cos_sin_apply_rotary_emb_new(input, input, self.freqs_cis)



In [4]:
m = MModule()
m = torch.compile(m, fullgraph=True)

/home/fsuser/miniconda3/envs/ncs-stable/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
m_old = MoldModuole()
m_old = torch.compile(m_old, fullgraph=True)

In [10]:
m_out = m(torch.rand(1, 1024, 1, 512))

In [6]:
m_old_out = m_old(torch.rand(1, 1024, 1, 512))